# Day 28: Agent Protocols (MCP, A2A) and Advanced Agents
**A sandboxed data agent, an MCP server on the current SDK used by an agent, MCP Tasks, signed A2A Agent Cards and agentic handoff**

Prepared by **Ahmed** | [linkedin.com/in/ahmed-al-imaad](https://www.linkedin.com/in/ahmed-al-imaad)

---
### How to use this notebook
* Run a cell with **Shift + Enter**.
* Some cells have two versions: **Option A (Google Colab)** is active, **Option B (VS Code / local)** is commented out.
  * In Colab: run as is.
  * In VS Code: comment Option A (select lines, press `Ctrl + /` or `Cmd + /`) and uncomment Option B.
* Cells marked **TODO** are for you to complete.
* Sections marked **NEW (v4)** were added in program v4.

## 1. Install
The MCP demo uses the **current** Python SDK (mcp 2.x). There is no `mcp<2` pin any more.

In [ ]:
# ===== Option A: Google Colab (active) =====
# Add your key once: click the key icon (Secrets) in the left sidebar, name it OPENAI_API_KEY, and allow notebook access.
!pip install -q openai tiktoken scikit-learn numpy pandas "mcp>=2" langchain langchain-core langchain-openai langgraph cryptography

# ===== Option B: VS Code / local =====
# pip install openai tiktoken scikit-learn numpy pandas "mcp>=2" langchain langchain-core langchain-openai langgraph cryptography
# Create a .env file (git-ignored) or set the variable in your terminal:  OPENAI_API_KEY=...
# No key? Everything still runs in OFFLINE demo mode with built-in stand-ins.
import importlib.metadata as md
print("mcp", md.version("mcp"), "| langchain", md.version("langchain"))

## Setup and API key
With an `OPENAI_API_KEY` the notebook calls a real model. Without one, it runs in **offline demo mode** using a tiny stand-in, so every cell still works and you can study the code. Offline scores show how the code works, not real model quality.

The model name comes from the `LLM_MODEL` environment variable. The default is the program's approved default; confirm it against the approved model list in `docs/MODEL_GUIDE.md`.

In [ ]:
%%writefile genai_helpers.py
"""Shared helpers for the GenAI days.
llm()   -> calls OpenAI when a key is available, otherwise a small OFFLINE stand-in so every cell still runs.
embed() -> OpenAI embeddings online, a hashing-based stand-in offline (same API, lower quality).
usage() -> token counts of the last llm() call (estimated offline).
"""
import json, os, re
import numpy as np

def _get_key():
    if os.getenv("OFFLINE") == "1":                 # force offline demo mode (used by tools/check_notebooks.py)
        return None
    try:
        from google.colab import userdata          # Colab Secrets
        return userdata.get("OPENAI_API_KEY")
    except Exception:
        return os.getenv("OPENAI_API_KEY")

OPENAI_API_KEY = _get_key()
ONLINE = bool(OPENAI_API_KEY)
# Default model: confirm against the approved model list (docs/MODEL_GUIDE.md). Override with LLM_MODEL.
MODEL = os.getenv("LLM_MODEL", "gpt-6.1-sol")
EMBED_MODEL = os.getenv("EMBED_MODEL", "text-embedding-3-small")
if ONLINE:
    from openai import OpenAI
    client = OpenAI(api_key=OPENAI_API_KEY)

WORD = re.compile(r"[a-z0-9]+")
LAST_USAGE = {"input_tokens": 0, "output_tokens": 0}

_ENC = []

def count_tokens(text):
    if not _ENC:
        try:
            import tiktoken
            _ENC.append(tiktoken.get_encoding("o200k_base"))
        except Exception:
            _ENC.append(None)                        # no tiktoken or no download: use the estimate
    if _ENC[0] is not None:
        return len(_ENC[0].encode(text))
    return max(1, round(len(text) / 4))              # rough rule: about 4 characters per token

def _words(t):
    # crude normalisation so "expires" matches "expire"
    return {w[:-1] if w.endswith("s") and len(w) > 4 else w for w in WORD.findall(t.lower()) if len(w) > 3}

def offline_llm(prompt, system=""):
    """A deliberately simple stand-in. It is NOT intelligent; it lets the code paths run without a key."""
    low = prompt.lower()
    if "context:" in low and "question:" in low:
        ctx = re.split(r"(?i)context:", prompt, 1)[1]
        ctx, q = re.split(r"(?i)question:", ctx, 1)
        sents = [s.strip() for s in re.split(r"(?<=[.!?])\s+", ctx.strip()) if s.strip()]
        best = max(sents, key=lambda s: len(_words(s) & _words(q)), default="")
        return best if best and _words(best) & _words(q) else "I don't know based on the provided context."
    if "classify" in low and "positive" in low:
        neg = sum(w in low for w in ["slow", "bad", "broken", "late", "angry", "worst", "refund"])
        pos = sum(w in low for w in ["great", "love", "fast", "thanks", "excellent", "happy"])
        return "negative" if neg > pos else "positive" if pos > neg else "neutral"
    if "classify" in low:                            # SmartDesk ticket categories
        last = low.strip().splitlines()[-1]
        for label, words in [("billing", ["bill", "charge", "refund", "invoice", "payment"]),
                             ("network", ["signal", "network", "slow", "outage", "data", "5g"]),
                             ("account", ["password", "login", "account", "sim", "plan"])]:
            if any(w in last for w in words):
                return label
        return "other"
    if "json" in low:
        text = prompt.strip().splitlines()[-1]
        return json.dumps({"summary": " ".join(text.split()[:12]), "offline": True})
    first = re.split(r"(?<=[.!?])\s+", prompt.strip())[0]
    return f"[offline demo] {first[:160]}"

def llm(prompt, system="You are a helpful assistant.", temperature=0.2, model=None, max_tokens=None):
    if ONLINE:
        kwargs = {"max_tokens": max_tokens} if max_tokens else {}
        r = client.chat.completions.create(
            model=model or MODEL, temperature=temperature,
            messages=[{"role": "system", "content": system}, {"role": "user", "content": prompt}], **kwargs)
        LAST_USAGE.update(input_tokens=r.usage.prompt_tokens, output_tokens=r.usage.completion_tokens)
        return r.choices[0].message.content
    out = offline_llm(prompt, system)
    LAST_USAGE.update(input_tokens=count_tokens(system + prompt), output_tokens=count_tokens(out))
    return out

def usage():
    return dict(LAST_USAGE)

def embed(texts):
    """Return an (n, d) array of unit-length vectors."""
    if isinstance(texts, str):
        texts = [texts]
    if ONLINE:
        r = client.embeddings.create(model=EMBED_MODEL, input=texts)
        v = np.array([d.embedding for d in r.data])
    else:
        from sklearn.feature_extraction.text import HashingVectorizer
        hv = HashingVectorizer(n_features=1024, ngram_range=(1, 2), alternate_sign=False, norm=None)
        v = hv.transform([t.lower() for t in texts]).toarray()
    n = np.linalg.norm(v, axis=1, keepdims=True)
    return v / np.where(n == 0, 1, n)

print("Mode:", f"ONLINE ({MODEL})" if ONLINE else "OFFLINE demo (add OPENAI_API_KEY to use a real model)")

In [ ]:
import importlib, genai_helpers
importlib.reload(genai_helpers)
from genai_helpers import llm, embed, usage, count_tokens, ONLINE, MODEL

## 2. A data analysis agent
The model writes pandas code. We check the code with the AST, then run it.

In [ ]:
import ast, json, re, sys, os, subprocess, tempfile
import numpy as np, pandas as pd

rng = np.random.default_rng(28)
tickets = pd.DataFrame({
    "month": np.repeat(["Sep", "Oct", "Nov"], 8),
    "city": ["Tokyo", "Osaka", "Bengaluru", "Pune"] * 6,
    "priority": rng.choice(["P1", "P2", "P3"], 24, p=[0.2, 0.5, 0.3]),
    "minutes_to_resolve": rng.integers(20, 600, 24),
})
DATA_DIR = tempfile.mkdtemp()
CSV = os.path.join(DATA_DIR, "tickets.csv")
tickets.to_csv(CSV, index=False)

BANNED_NAMES = {"open", "exec", "eval", "compile", "__import__", "globals", "locals", "getattr", "setattr",
                "os", "sys", "subprocess", "input", "breakpoint"}

def check_code(code):
    tree = ast.parse(code)
    for node in ast.walk(tree):
        if isinstance(node, (ast.Import, ast.ImportFrom)):
            return "imports are not allowed"
        if isinstance(node, ast.Name) and node.id in BANNED_NAMES:
            return f"name '{node.id}' is not allowed"
        if isinstance(node, ast.Attribute) and node.attr.startswith("_"):
            return "private attributes are not allowed"
    return None

TEMPLATES = {   # offline stand-in for the model writing pandas code
    "most p1": "result = df[df.priority == 'P1'].groupby('city').size().idxmax()",
    "average resolution by month": "result = df.groupby('month', sort=False)['minutes_to_resolve'].mean().round(1).to_dict()",
    "slowest city": "result = df.groupby('city')['minutes_to_resolve'].median().idxmax()",
}

def write_code(question):
    if ONLINE:
        r = llm(f"Write pandas code using an existing DataFrame df with columns {list(tickets.columns)}. "
                f"Store the answer in a variable named result. No imports. Code only.\nQuestion: {question}", temperature=0)
        return r.replace("```python", "").replace("```", "").strip()
    return next((c for k, c in TEMPLATES.items() if k in question.lower()), "result = len(df)")
print(check_code("result = df.head()"), "|", check_code("import os"))

## 3. NEW (v4): Sandboxed code execution
Each run happens in a **separate process** with a time limit, CPU and memory limits, an empty environment and a throwaway working folder. This is the offline fallback. On your laptop, the Docker cell below adds **no network** and a read-only file system.

In [ ]:
def run_sandboxed(code, csv_path=CSV, timeout=5):
    problem = check_code(code)
    if problem:
        return f"BLOCKED: {problem}"
    job = (f"import pandas as pd\ndf = pd.read_csv({csv_path!r})\n"
           f"{code}\nprint(repr(result))\n")

    def limits():                                   # POSIX only (Colab, Linux, macOS)
        try:
            import resource
            resource.setrlimit(resource.RLIMIT_CPU, (timeout, timeout))
            resource.setrlimit(resource.RLIMIT_AS, (2_000_000_000, 2_000_000_000))
        except Exception:
            pass

    with tempfile.TemporaryDirectory() as work:
        try:
            p = subprocess.run([sys.executable, "-I", "-c", job], capture_output=True, text=True, timeout=timeout,
                               cwd=work, env={"OPENBLAS_NUM_THREADS": "1", "PATH": ""},
                               preexec_fn=limits if os.name == "posix" else None)
        except subprocess.TimeoutExpired:
            return f"KILLED: time limit {timeout}s"
    if p.returncode < 0:
        return f"KILLED: resource limit (signal {-p.returncode})"
    if p.returncode != 0:
        return f"ERROR: {p.stderr.strip().splitlines()[-1] if p.stderr.strip() else p.returncode}"
    return p.stdout.strip()

for q in ["Which city has the most P1 tickets?", "Average resolution by month?", "Which is the slowest city?"]:
    code = write_code(q)
    print(f"Q: {q}\n   code: {code}\n   result: {run_sandboxed(code)}\n")

for evil in ["import os\nresult = os.listdir('.')", "result = open('/etc/passwd').read()", "result = df.__class__.__bases__",
             "result = sum(i * i for i in range(10**12))"]:
    print(repr(evil[:40]), "->", run_sandboxed(evil, timeout=3))

### The container version (Docker Desktop on your laptop)
This cell is skipped in the offline check and in Colab. It runs the same job with no network, a read-only file system and resource limits.

In [ ]:
# ===== Option B only: VS Code / local with Docker Desktop =====
job_dir = tempfile.mkdtemp()
with open(os.path.join(job_dir, "analysis.py"), "w") as f:
    f.write("import pandas as pd\ndf = pd.read_csv('/data/tickets.csv')\n"
            "print(df.groupby('city')['minutes_to_resolve'].median().idxmax())\n")
dockerfile = "FROM python:3.12-slim
RUN pip install --no-cache-dir pandas
USER 1000:1000
"
subprocess.run(["docker", "build", "-t", "smartdesk-sandbox", "-"], input=dockerfile, text=True, check=True)
cmd = ["docker", "run", "--rm", "--network", "none", "--read-only", "--memory", "256m", "--cpus", "0.5",
       "--pids-limit", "64", "-v", f"{DATA_DIR}:/data:ro", "-v", f"{job_dir}:/job:ro",
       "smartdesk-sandbox", "timeout", "10", "python", "/job/analysis.py"]
print(" ".join(cmd))
print(subprocess.run(cmd, capture_output=True, text=True).stdout)

Note: the image is built once with pandas pre-installed, because with `--network none` nothing can be downloaded at run time.

## 4. NEW (v4): An MCP server on the current SDK
In mcp 2.x, `FastMCP` is now `MCPServer` (`from mcp.server.mcpserver import MCPServer`). We write the server to a file so it can also run over stdio.

In [ ]:
%%writefile mcp_server.py
"""SmartDesk ticket tools as an MCP server (mcp 2.x)."""
import json, re
from mcp.server.mcpserver import MCPServer

mcp = MCPServer("smartdesk-tickets", version="1.0",
                instructions="Read-only SmartDesk ticket tools. Ticket ids look like T-1001.")
TICKETS = {"T-1001": {"status": "open", "priority": "P2", "category": "network", "city": "Osaka"},
           "T-1002": {"status": "open", "priority": "P1", "category": "billing", "city": "Tokyo"},
           "T-1003": {"status": "closed", "priority": "P3", "category": "account", "city": "Pune"}}
AUDIT = []

@mcp.tool()
def get_ticket(ticket_id: str) -> str:
    """Look up one SmartDesk ticket by id, for example T-1001. Returns status, priority and category."""
    AUDIT.append(("get_ticket", ticket_id))
    if not re.fullmatch(r"T-\d{4}", ticket_id):
        return json.dumps({"error": "ticket id must look like T-1234"})
    return json.dumps(TICKETS.get(ticket_id, {"error": "not found"}))

@mcp.tool()
def list_open_tickets(limit: int = 10) -> str:
    """List open ticket ids, newest first. limit is 1 to 20."""
    AUDIT.append(("list_open_tickets", limit))
    ids = [k for k, v in TICKETS.items() if v["status"] == "open"][: max(1, min(limit, 20))]
    return json.dumps(ids)

@mcp.resource("policy://sla")
def sla_policy() -> str:
    """The SmartDesk SLA policy."""
    return "P1 tickets must be resolved within 4 hours, P2 within 1 day, P3 within 3 days."

@mcp.prompt()
def apology_email(customer: str, issue: str) -> str:
    """A template for a short apology email."""
    return f"Write a short, warm apology email to {customer} about: {issue}. Offer one concrete next step."

if __name__ == "__main__":
    mcp.run()          # stdio; for remote use: mcp.run("streamable-http", stateless_http=True)

### Connect with `mcp.Client` (in-process)
Passing the server object connects in-process: ideal for tests and offline runs. Notebooks support top-level `await`.

In [ ]:
import importlib, mcp_server
importlib.reload(mcp_server)
from mcp import Client

async with Client(mcp_server.mcp) as c:
    print("protocol:", c.protocol_version)
    tools = await c.list_tools()
    print("tools:", [t.name for t in tools.tools])
    print("schema:", tools.tools[0].input_schema)
    r = await c.call_tool("get_ticket", {"ticket_id": "T-1001"})
    print("get_ticket ->", r.content[0].text)
    r = await c.call_tool("list_open_tickets", {"limit": 5})
    print("list_open_tickets ->", r.content[0].text)
    bad = await c.call_tool("get_ticket", {"ticket_id": "DROP TABLE"})
    print("invalid input ->", bad.content[0].text)
    print("resources:", [str(x.uri) for x in (await c.list_resources()).resources])
    print("sla:", (await c.read_resource("policy://sla")).contents[0].text)
    p = await c.get_prompt("apology_email", {"customer": "Mei", "issue": "slow mobile data"})
    print("prompt:", p.messages[0].content.text)

### Connect over stdio (a real separate process)

In [ ]:
from mcp import StdioServerParameters

params = StdioServerParameters(command=sys.executable, args=["mcp_server.py"])
async with Client(params) as c:
    r = await c.call_tool("get_ticket", {"ticket_id": "T-1002"})
    print("stdio:", c.protocol_version, "->", r.content[0].text)

## 5. NEW (v4): Use the MCP server from an agent
We wrap each MCP tool as a LangChain tool and give them to a `create_agent` assistant. Offline, a small rule-based model decides the calls.

In [ ]:
from typing import Any
from langchain.agents import create_agent
from langchain_core.language_models import BaseChatModel
from langchain_core.messages import AIMessage, HumanMessage, ToolMessage
from langchain_core.outputs import ChatGeneration, ChatResult
from langchain_core.tools import StructuredTool

class OfflineTicketModel(BaseChatModel):
    """Rule-based stand-in. NOT intelligent: it lets the agent loop run without a key."""
    tool_names: list = []

    @property
    def _llm_type(self):
        return "offline-demo"

    def bind_tools(self, tools, **kwargs):
        return self.model_copy(update={"tool_names": [t.name for t in tools]})

    def _generate(self, messages, stop=None, run_manager=None, **kwargs):
        user = next(m.content for m in reversed(messages) if isinstance(m, HumanMessage))
        used = [m for m in messages if isinstance(m, ToolMessage)]
        tid = re.search(r"T-\d{4}", user)
        if tid and not used and "get_ticket" in self.tool_names:
            msg = AIMessage(content="", tool_calls=[{"name": "get_ticket", "args": {"ticket_id": tid.group(0)}, "id": "c1"}])
        elif not tid and not used and "list_open_tickets" in self.tool_names:
            msg = AIMessage(content="", tool_calls=[{"name": "list_open_tickets", "args": {"limit": 5}, "id": "c1"}])
        else:
            msg = AIMessage(content=f"[offline demo] From the ticket system: {used[-1].content if used else 'nothing found'}")
        return ChatResult(generations=[ChatGeneration(message=msg)])

def chat_model():
    if ONLINE:
        from langchain_openai import ChatOpenAI
        return ChatOpenAI(model=MODEL, api_key=genai_helpers.OPENAI_API_KEY)
    return OfflineTicketModel()

def mcp_tools_for(client, listed):
    def make(t):
        async def call(**kwargs):
            r = await client.call_tool(t.name, kwargs)
            print(f"  [mcp] {t.name} {kwargs}")
            return r.content[0].text
        return StructuredTool.from_function(coroutine=call, name=t.name, description=t.description or "",
                                            args_schema=t.input_schema)
    return [make(t) for t in listed.tools]

import genai_helpers
async with Client(mcp_server.mcp) as c:
    tools = mcp_tools_for(c, await c.list_tools())
    agent = create_agent(chat_model(), tools=tools, system_prompt="You are a SmartDesk agent. Use the ticket tools.")
    for q in ["What is the status of T-1002?", "Which tickets are still open?"]:
        out = await agent.ainvoke({"messages": [{"role": "user", "content": q}]})
        print("Q:", q, "\nA:", out["messages"][-1].content, "\n")

### Add the same server to OpenAI Codex
Codex can use MCP servers too. Add a read-only server for local development in your Codex configuration (`~/.codex/config.toml`) or with the `codex mcp add` command. Check the current Codex documentation for your version.

```toml
[mcp_servers.smartdesk]
command = "python"
args = ["mcp_server.py"]
```

Rules: only reviewed servers, read-only tools for Codex, no customer data or secrets in a dev setup.

## 6. NEW (v4): MCP 2026-07-28: stateless, OAuth 2.1 and Tasks
* **Stateless:** the client above reported protocol `2026-07-28`; each request stands alone, so a remote server can run behind a load balancer with `stateless_http=True`.
* **OAuth 2.1:** remote servers verify a bearer token on every request (code below, not run here).
* **Tasks:** long-running calls return a task the client polls. mcp 2.3.0 ships the `Task` types; we simulate the lifecycle with them.

In [ ]:
# Shown, not run: a remote MCP server that checks OAuth 2.1 bearer tokens (mcp 2.x).
REMOTE_SERVER_EXAMPLE = """
from mcp.server.auth.provider import AccessToken
from mcp.server.auth.settings import AuthSettings
from mcp.server.mcpserver import MCPServer

class MyVerifier:
    async def verify_token(self, token: str) -> AccessToken | None:
        claims = check_signature_and_expiry(token)        # your identity provider's library
        if not claims:
            return None
        return AccessToken(token=token, client_id=claims["client_id"], scopes=claims["scope"].split(),
                           expires_at=claims["exp"], resource=claims["aud"])

mcp = MCPServer("smartdesk", token_verifier=MyVerifier(),
                auth=AuthSettings(issuer_url="https://login.example.com",
                                  resource_server_url="https://mcp.example.com",
                                  required_scopes=["tickets:read"], validate_token_resource=True))
mcp.run("streamable-http", stateless_http=True)
"""
print(REMOTE_SERVER_EXAMPLE)

In [ ]:
import uuid, datetime as dt
import mcp_types as mt

class TaskBoard:
    """Simulated MCP task store using the SDK's Task type."""
    def __init__(self):
        self.tasks, self.results = {}, {}

    def create(self, ttl_ms=600_000):
        now = dt.datetime.now(dt.timezone.utc).isoformat()
        t = mt.Task(task_id=uuid.uuid4().hex[:4], status="working", created_at=now, last_updated_at=now,
                    ttl=ttl_ms, poll_interval=1000)
        self.tasks[t.task_id] = t
        return t

    def update(self, task_id, status, result=None):
        t = self.tasks[task_id].model_copy(update={"status": status,
                                                   "last_updated_at": dt.datetime.now(dt.timezone.utc).isoformat()})
        self.tasks[task_id] = t
        if result is not None:
            self.results[task_id] = result

board = TaskBoard()
task = board.create()
history = [task.status]
for step in range(3):                       # the slow export makes progress
    if step == 2:
        board.update(task.task_id, "completed", result="120 tickets exported to tickets_nov.csv")
    history.append(board.tasks[task.task_id].status)
print(f"task {task.task_id}: {' -> '.join(history)}")
print("result:", board.results[task.task_id])

other = board.create()
board.update(other.task_id, "cancelled")
print(f"task {other.task_id}:", board.tasks[other.task_id].status, "| allowed statuses:", list(mt.TaskStatus.__args__))

## 7. NEW (v4): A2A v1.0 and signed Agent Cards
Two small FastAPI agents. The refund agent publishes a **signed** Agent Card; the support agent verifies the signature before delegating a task. This is a simplified form: follow the A2A v1.0 specification for the exact card fields and signature format.

In [ ]:
import base64, logging
logging.getLogger("httpx").setLevel(logging.WARNING)
logging.getLogger("httpx2").setLevel(logging.WARNING)
from cryptography.hazmat.primitives.asymmetric.ed25519 import Ed25519PrivateKey
from cryptography.exceptions import InvalidSignature
from fastapi import FastAPI
from fastapi.testclient import TestClient
from pydantic import BaseModel

def canonical(card):
    body = {k: v for k, v in card.items() if k != "signatures"}
    return json.dumps(body, sort_keys=True, separators=(",", ":")).encode()

key = Ed25519PrivateKey.generate()          # the refund team's signing key
REGISTRY = {"refund-team-key-1": key.public_key()}   # approved agents and their public keys

CARD = {"name": "Refund Agent", "description": "Calculates late-delivery credits.", "version": "1.0",
        "url": "http://refund-agent.local/a2a", "skills": [{"id": "calculate_credit", "description": "Credit for minutes of outage"}]}
CARD["signatures"] = [{"kid": "refund-team-key-1", "alg": "EdDSA",
                       "signature": base64.urlsafe_b64encode(key.sign(canonical(CARD))).decode()}]

def verify(card):
    try:
        s = card["signatures"][0]
        REGISTRY[s["kid"]].verify(base64.urlsafe_b64decode(s["signature"]), canonical(card))
        return True
    except (KeyError, IndexError, InvalidSignature):
        return False

refund_agent = FastAPI(title="Refund Agent")

class TaskRequest(BaseModel):
    skill: str
    message: dict

@refund_agent.get("/.well-known/agent-card.json")
def agent_card():
    return CARD

@refund_agent.post("/tasks")
def create_task(req: TaskRequest):
    task_id = uuid.uuid4().hex[:8]
    if req.skill != "calculate_credit":
        return {"id": task_id, "status": "failed", "error": f"unknown skill {req.skill}"}
    credit = round(min(req.message["outage_minutes"] / 60 * 50, 2000), 2)    # 50 per hour, max 2,000
    return {"id": task_id, "status": "completed", "artifacts": [{"type": "data", "data": {"credit": credit}}]}

remote = TestClient(refund_agent)

def support_agent(outage_minutes, card=None):
    card = card or remote.get("/.well-known/agent-card.json").json()
    if not verify(card):
        return "Card verification failed: refusing to delegate."
    print("verified:", card["name"], "skills:", [s["id"] for s in card["skills"]])
    task = remote.post("/tasks", json={"skill": "calculate_credit", "message": {"outage_minutes": outage_minutes}}).json()
    print("task:", task["id"], task["status"])
    return f"Sorry for the outage. A credit of {task['artifacts'][0]['data']['credit']} has been added."

print(support_agent(180))
forged = json.loads(json.dumps(CARD))
forged["url"] = "http://attacker.example/a2a"
print(support_agent(180, card=forged))

## 8. Agentic handoff

In [ ]:
from typing import Literal

class Handoff(BaseModel):
    from_agent: str
    to: str
    summary: str
    facts: dict
    steps_tried: list[str]
    reason: str
    suggested_next_step: str
    urgency: Literal["low", "medium", "high"]

h = Handoff(
    from_agent="smartdesk-bot", to="human-billing-team",
    summary="Customer Mei had no mobile data for 9 hours in Osaka and wants a credit; she is upset.",
    facts={"ticket_id": "T-1001", "outage_minutes": 540, "calculated_credit": 450.0},
    steps_tried=["get_ticket -> open, P2, network", "Refund Agent calculate_credit -> 450.0"],
    reason="Customer asked for a larger credit than the bot may approve",
    suggested_next_step="Review the outage report and approve a credit up to 1,000",
    urgency="high",
)
print(h.model_dump_json(indent=2))

### TODO 1 (SmartDesk step 24): add an `update_status(ticket_id, status)` tool that is write-only for agents with an approval, and a `search_faq(query)` tool. Call both from the agent.
### TODO 2: add a row limit (refuse data over 100,000 rows) and block `while` loops in `check_code`.

In [ ]:
# TODO: write your code here

## 9. Check yourself
1. Which protocol version did the client report?
2. What stopped the never-ending loop in the sandbox?
3. Why did the forged card fail verification?
4. Where would you set OAuth scopes for a remote MCP server?

---
**Yoi shuumatsu wo (良い週末を): *Have a great weekend***

Prepared by **Ahmed** | [linkedin.com/in/ahmed-al-imaad](https://www.linkedin.com/in/ahmed-al-imaad)